# Fine-tune Qwen3-8B on the Vedaz Astrology Dataset (QLoRA + Unsloth)

This notebook trains, exports, and smoke-tests the astrology-assistant fine-tune end to end on a free Colab GPU. It clones the project straight from GitHub (`soulrahulrk/model-finetune`) so there is **no manual file uploading** — just run the cells top to bottom.

**Before running anything:** `Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU` (or better, if you have Colab Pro: L4 / A100), then `Save`.

What this notebook does, in order:
1. Check the GPU
2. Install Unsloth + training dependencies (Colab-safe — does **not** touch Colab's preinstalled torch)
3. Clone the project repo
4. Sanity-check the data/config
5. Train (QLoRA, ~15-30 min on a T4 for this dataset size) ← **screenshot moment**
6. Merge the LoRA adapter + export
7. Run a real inference test ← **screenshot moment**
8. Zip and download your results

Background on every design choice here (why Qwen3, why QLoRA, why these hyperparameters) is in `docs/02_finetuning_strategy.md` in the repo.

## 1. Check the GPU

In [24]:
!nvidia-smi
print()
print("If the above says 'command not found' or shows no GPU, go to Runtime -> Change runtime type -> GPU, then Save, then Runtime -> Restart session, and re-run from the top.")

Thu Jul  2 18:13:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P0             29W /   70W |     155MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install dependencies

We deliberately install with `--no-deps` for the core stack and do **not** run `pip install --upgrade torch`. Colab's preinstalled torch build is already matched to its CUDA driver; force-upgrading it (or letting pip resolve a fresh torch as a side effect of another package) is the single most common cause of Unsloth/bitsandbytes breaking on Colab with a confusing CUDA-mismatch error. This cell takes 2-4 minutes.

In [25]:
%%capture
!pip install --no-deps bitsandbytes accelerate peft trl triton cut_cross_entropy unsloth_zoo xformers
!pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" huggingface_hub hf_transfer pyyaml
!pip install --no-deps unsloth

# Colab preinstalls an old torchao (0.10.0) that's incompatible with recent peft's LoRA-layer
# dispatch code, which does an eager version check and raises ImportError even though we never
# actually use torchao (our quantization is bitsandbytes NF4, not torchao). Removing it entirely
# makes peft's is_torchao_available() cleanly report "not available" and skip that code path.
!pip uninstall -y torchao

In [26]:
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

import unsloth
print("unsloth:", unsloth.__version__)

torch: 2.11.0+cu128 | CUDA available: True
GPU: Tesla T4 | VRAM: 15.6 GB
unsloth: 2026.6.9


**If the install cell above errors:** run `Runtime -> Restart session`, then re-run just this install cell again on its own before continuing (a fresh runtime resolves the vast majority of Colab dependency-resolution hiccups).

## 3. Clone the project

Pulls `training_config.yaml`, `train_unsloth.py`, `merge_and_export.py`, `infer.py`, and the already-prepared `data/processed/train.jsonl` / `val.jsonl` (139 train / 16 val examples) straight from your GitHub repo — nothing to upload by hand.

In [27]:
%env REPO_URL=https://github.com/soulrahulrk/model-finetune.git
%env REPO_DIR=/content/model-finetune

!rm -rf "$REPO_DIR"
!git clone --depth 1 "$REPO_URL" "$REPO_DIR"

%env TRAIN_DIR=/content/model-finetune/src/training
%env INFER_DIR=/content/model-finetune/src/inference
%env DATA_DIR=/content/model-finetune/data/processed

# Also define these as plain Python variables (not just OS env vars) — IPython magics like
# %cd only expand $NAME against Python variables in the notebook namespace, NOT against
# %env-only OS environment variables. Having both means $VAR works in both !shell lines
# and in magics.
import os
REPO_DIR = os.environ["REPO_DIR"]
TRAIN_DIR = os.environ["TRAIN_DIR"]
INFER_DIR = os.environ["INFER_DIR"]
DATA_DIR = os.environ["DATA_DIR"]

!echo "--- src/training ---" && ls -la "$TRAIN_DIR"
!echo "--- data/processed ---" && ls -la "$DATA_DIR"
!echo "train.jsonl lines:" && wc -l "$DATA_DIR/train.jsonl"
!echo "val.jsonl lines:" && wc -l "$DATA_DIR/val.jsonl"

env: REPO_URL=https://github.com/soulrahulrk/model-finetune.git
env: REPO_DIR=/content/model-finetune
Cloning into '/content/model-finetune'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 59 (delta 2), reused 59 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 267.67 KiB | 6.53 MiB/s, done.
Resolving deltas: 100% (2/2), done.
env: TRAIN_DIR=/content/model-finetune/src/training
env: INFER_DIR=/content/model-finetune/src/inference
env: DATA_DIR=/content/model-finetune/data/processed
--- src/training ---
total 24
drwxr-xr-x 2 root root 4096 Jul  2 18:14 .
drwxr-xr-x 6 root root 4096 Jul  2 18:14 ..
-rw-r--r-- 1 root root 2472 Jul  2 18:14 merge_and_export.py
-rw-r--r-- 1 root root 2312 Jul  2 18:14 training_config.yaml
-rw-r--r-- 1 root root 7327 Jul  2 18:14 train_unsloth.py
--- data/processed ---
total 392
drwxr-xr-x 2 root root   4096 Jul  2 18:14 .
drwxr-xr-x 6 roo

**If the clone fails or `data/processed/train.jsonl` is missing:** the repo may not have that file pushed. Use the fallback cell below to upload `train.jsonl`, `val.jsonl`, `training_config.yaml`, and `train_unsloth.py` manually instead (from the `upload_to_colab/` folder on your own machine), then skip the clone cell above.

In [28]:
# FALLBACK — only run this if the git clone cell above did not work.
# Uploads open a file picker; select train.jsonl, val.jsonl, training_config.yaml,
# train_unsloth.py, merge_and_export.py, infer.py from upload_to_colab/ on your computer.

# import os
# from google.colab import files
# os.makedirs("/content/model-finetune/src/training", exist_ok=True)
# os.makedirs("/content/model-finetune/src/inference", exist_ok=True)
# os.makedirs("/content/model-finetune/data/processed", exist_ok=True)
# %cd /content/model-finetune/src/training
# uploaded = files.upload()  # pick training_config.yaml, train_unsloth.py, merge_and_export.py
# %cd /content/model-finetune/data/processed
# uploaded = files.upload()  # pick train.jsonl, val.jsonl
# %cd /content/model-finetune/src/inference
# uploaded = files.upload()  # pick infer.py
# %env TRAIN_DIR=/content/model-finetune/src/training
# %env INFER_DIR=/content/model-finetune/src/inference
# %env DATA_DIR=/content/model-finetune/data/processed

## 4. (Optional) Hugging Face login

Not required — `Qwen/Qwen3-8B` is open-weight, no gated access needed. Only run this if you hit a rate-limit/auth error downloading the base model, or if you want to push results to your own HF account later.

In [29]:
# from huggingface_hub import login
# login()  # opens a token-paste prompt; get a token at https://huggingface.co/settings/tokens

## 5. Train (QLoRA via Unsloth)

Default config: Qwen3-8B, rank-16 QLoRA, batch size 2 x grad-accum 4, 3 epochs, 4096 max sequence length — fits in ~7-9GB VRAM, comfortable on a 16GB T4. Expect **15-30 minutes** on a T4 for this dataset (139 train examples x 3 epochs).

**If you hit `CUDA out of memory` on a smaller GPU:** re-run this cell with the override flags shown commented below.

In [30]:
!cd "$TRAIN_DIR" && python train_unsloth.py --config training_config.yaml

# If you hit CUDA OOM, cancel (Runtime -> Interrupt execution) and instead run:
# !cd "$TRAIN_DIR" && python train_unsloth.py --config training_config.yaml \
#     --per_device_train_batch_size 1 --gradient_accumulation_steps 8

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[1/7] Loading base model in 4-bit: Qwen/Qwen3-8B
==((====))==  Unsloth 2026.6.9: Fast Qwen3 patching. Transformers: 5.12.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are

**📸 SCREENSHOT NOW.** While the table above is printing `loss` decreasing epoch over epoch, take a full-screen screenshot as your training evidence. Save it as `training_screenshot.png`.

### Sanity check: did training actually finish?

Run this before section 6. It checks for the adapter file that only exists if training completed successfully — catches the case where the training cell above was interrupted or disconnected partway through (e.g. it stopped mid-way through loading the base model, before the training loop even started) and tells you clearly instead of letting you hit a confusing error two cells later.

In [31]:
import os

adapter_config = os.path.join(os.environ["TRAIN_DIR"], "outputs", "qwen3-8b-vedaz-adapter", "adapter_config.json")
if os.path.exists(adapter_config):
    print("OK: training finished and the adapter was saved at", adapter_config)
    print("Safe to continue to section 6 (Merge the LoRA adapter and export).")
else:
    print("STOP: no adapter_config.json found at", adapter_config)
    print("This means the training cell above (section 5) did NOT finish — it was either")
    print("interrupted, or is still running, or errored out partway through.")
    print("Scroll up and check the FULL output of the training cell for either:")
    print("  (a) a 'Training complete. Metrics: {...}' line followed by 'Saving LoRA adapter...', or")
    print("  (b) an actual Python error/traceback.")
    print("If neither is present, re-run section 5 and let it run uninterrupted until it")
    print("prints 'Next step: python merge_and_export.py ...' before continuing to section 6.")

STOP: no adapter_config.json found at /content/model-finetune/src/training/outputs/qwen3-8b-vedaz-adapter/adapter_config.json
This means the training cell above (section 5) did NOT finish — it was either
interrupted, or is still running, or errored out partway through.
Scroll up and check the FULL output of the training cell for either:
  (a) a 'Training complete. Metrics: {...}' line followed by 'Saving LoRA adapter...', or
  (b) an actual Python error/traceback.
If neither is present, re-run section 5 and let it run uninterrupted until it
prints 'Next step: python merge_and_export.py ...' before continuing to section 6.


## 6. Merge the LoRA adapter and export

Merges the trained adapter back into the base model (fp16 safetensors — what you'd point vLLM at) and saves the adapter alone (small, easy to download). GGUF export is skipped by default since it's slow; remove `--skip-gguf` if you want a llama.cpp/Ollama-ready quantized file too.

In [32]:
!cd "$TRAIN_DIR" && python merge_and_export.py --config training_config.yaml --skip-gguf

!echo "--- outputs/ ---" && ls -la "$TRAIN_DIR/outputs/"
!echo "--- adapter dir ---" && ls -la "$TRAIN_DIR/outputs/qwen3-8b-vedaz-adapter/"

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading base model + adapter from ./outputs/qwen3-8b-vedaz-adapter
Traceback (most recent call last):
  File "/content/model-finetune/src/training/merge_and_export.py", line 67, in <module>
    main()
  File "/content/model-finetune/src/training/merge_and_export.py", line 38, in main
    model, tokenizer = FastLanguageModel.from_pretrained(
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/unsloth/models/loader.py", line 560, in from_pretrained
    model_types = get_transformers_model_type(
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/unsloth_zoo/hf_utils.py", line 110, in get_transformers_model_type
    raise RuntimeError(
RuntimeError: Unsloth: No config file found - are you sure the `model_name` is correct?
If you're using a model on your loca

## 7. Quick inference smoke test

Loads the adapter (Unsloth auto-detects `adapter_config.json` and pulls the base model automatically, so pointing `--model` straight at the adapter directory works without a full merge) and asks it a real astrology question in Hinglish.

In [33]:
!cd "$INFER_DIR" && python infer.py \
  --model "$TRAIN_DIR/outputs/qwen3-8b-vedaz-adapter" \
  --prompt "Namaste, meri shadi kab hogi? Date of birth 14 March 1998 hai."

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Traceback (most recent call last):
  File "/content/model-finetune/src/inference/infer.py", line 173, in <module>
    main()
  File "/content/model-finetune/src/inference/infer.py", line 157, in main
    model, tokenizer, backend = load_model(args.model, args.max_seq_length)
                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/model-finetune/src/inference/infer.py", line 45, in load_model
    model, tokenizer = FastLanguageModel.from_pretrained(
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/unsloth/models/loader.py", line 560, in from_pretrained
    model_types = get_transformers_model_type(
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/unsloth_zoo/hf_utils.py", line 110, in get_transformers_model_type
 

**📸 SCREENSHOT NOW.** Once the model replies in Hindi/Hinglish/English in the astrologer persona, take a full-screen screenshot. Save it as `inference_screenshot.png`.

Want to try the merged (non-adapter) model instead, or test other prompts / interactive chat?

In [34]:
# Test against the fully merged fp16 model instead of the adapter:
# !cd "$INFER_DIR" && python infer.py --model "$TRAIN_DIR/outputs/qwen3-8b-vedaz-merged-fp16" \
#   --prompt "Mera career kab tak stabilize hoga?"

# Interactive chat (keeps asking until you type 'exit'):
# !cd "$INFER_DIR" && python infer.py --model "$TRAIN_DIR/outputs/qwen3-8b-vedaz-adapter" --interactive

## 8. Zip and download your results

Zips the LoRA adapter (small, a few hundred MB) for direct browser download. The full merged fp16 model (~16GB) is too large for a reliable browser download — use the Google Drive cell below instead if you need it off of Colab.

In [35]:
import os
import shutil
from google.colab import files

adapter_dir = os.path.join(os.environ["TRAIN_DIR"], "outputs", "qwen3-8b-vedaz-adapter")
zip_path = shutil.make_archive("/content/qwen3-8b-vedaz-adapter", "zip", adapter_dir)
print("Zipped:", zip_path)

files.download(zip_path)

FileNotFoundError: [Errno 2] No such file or directory: '/content/model-finetune/src/training/outputs/qwen3-8b-vedaz-adapter'

In [ ]:
# OPTIONAL: save the full merged fp16 model (~16GB) to your Google Drive instead of downloading.
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p "/content/drive/MyDrive/vedaz-finetune-outputs"
# !cp -r $TRAIN_DIR/outputs/qwen3-8b-vedaz-merged-fp16 "/content/drive/MyDrive/vedaz-finetune-outputs/"
# !cp -r $TRAIN_DIR/outputs/qwen3-8b-vedaz-adapter "/content/drive/MyDrive/vedaz-finetune-outputs/"
# print("Saved to Google Drive: MyDrive/vedaz-finetune-outputs/")

## Troubleshooting

| Symptom | Fix |
|---|---|
| Install cell errors | `Runtime -> Restart session`, re-run the install cell alone first |
| `CUDA out of memory` during training | Use the override flags in the training cell (`--per_device_train_batch_size 1 --gradient_accumulation_steps 8`) |
| `git clone` fails / files missing | Use the commented FALLBACK upload cell in section 3 |
| Session disconnects mid-training | Free Colab has a runtime time limit (~12h, less if idle); Colab Pro is more reliable for longer runs. Re-run from section 3 onward — training resumes from scratch since checkpoints aren't restored automatically in this notebook |
| Want to change model size for a smaller GPU | Edit `model.base_model` in `training_config.yaml` (uploaded/cloned copy) from `Qwen/Qwen3-8B` to `Qwen/Qwen3-4B` before running section 5 |

Full written explanations of every hyperparameter and design decision: `docs/02_finetuning_strategy.md` in the repo.